In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

source_path = "/Volumes/workspace/ibm/v9/sales_source_1500.csv"

bronze_table = "workspace.default.capstone_bronze_sales9"

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(source_path)
)

print("Source record count:", df.count())

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(bronze_table)

print("Bronze table created:", bronze_table)

In [0]:
df.show(5)

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

source_path = "/Volumes/workspace/ibm/v9/"
checkpoint_path = "/Volumes/workspace/ibm/v10/checkpoints/bronze_sales9"
bronze_table = "workspace.default.capstone_bronze_sales9"

# Remove stale checkpoint from previous failed runs
try:
    dbutils.fs.rm(checkpoint_path, recurse=True)
except Exception:
    pass

stream_df = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .option("header", "true")
         .option("cloudFiles.inferColumnTypes", "true")
         .option("cloudFiles.schemaLocation", checkpoint_path + "/schema")
         .load(source_path)
)

query = (
    stream_df.writeStream
             .format("delta")
             .option("checkpointLocation", checkpoint_path)
             .option("mergeSchema", "true")
             .outputMode("append")
             .trigger(availableNow=True)
             .toTable(bronze_table)
)

query.awaitTermination()

print(f"Bronze table loaded successfully: {bronze_table}")